In [0]:
-- Team - Goals by Game Time
-- Goals for and against in five-minute blocks of each period.
-- Goal and shot totals come from play-by-play and leave out shootouts.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,       -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,  -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type       -- 2 = regular season, 3 = playoffs
),
team_games AS (                          -- one row per team per finished game
  SELECT g.game_id, g.season, g.game_date, g.date_key, g.last_period_type,
         t.team_id, t.abbrev AS team, o.abbrev AS opponent,
         t.team_id = g.home_team_id AS is_home, g.home_team_id,
         COALESCE(t.abbrev = pr.team, FALSE) AS is_selected,
         IF(t.team_id = g.home_team_id, g.home_final_score, g.away_final_score) AS final_gf,
         IF(t.team_id = g.home_team_id, g.away_final_score, g.home_final_score) AS final_ga,
         CASE WHEN IF(t.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.gold.dim_game g
  JOIN nhl.gold.dim_team t ON t.team_id IN (g.home_team_id, g.away_team_id)
  JOIN nhl.gold.dim_team o ON o.team_id = IF(t.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.is_final
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
),
team_events AS (                         -- plays in those games, from that team's side
  SELECT tg.team, tg.team_id, tg.opponent, tg.season, tg.game_date, tg.is_home, tg.result,
         IF(f.event_owner_team_id = tg.team_id, 'for', 'against') AS side,
         IF(f.event_owner_team_id = tg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         br.team_id AS blocker_team_id,
         f.*
  FROM nhl.gold.fact_event f
  JOIN team_games tg ON f.game_id = tg.game_id
  LEFT JOIN nhl.silver.game_rosters br                -- blocked shots belong to the shooter's team,
         ON br.game_id = f.game_id AND br.player_id = f.blocking_player_id   -- so credit blocks by roster
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
)
SELECT CASE WHEN period_type = 'OT' THEN 'OT'
            ELSE CONCAT('P', period_number, ' ',
                        LPAD(CAST(FLOOR(CAST(substr(time_in_period, 1, 2) AS INT) / 5) * 5 AS STRING), 2, '0'),
                        '-',
                        LPAD(CAST(FLOOR(CAST(substr(time_in_period, 1, 2) AS INT) / 5) * 5 + 5 AS STRING), 2, '0'),
                        ' min') END                   AS game_time,
       COUNT_IF(side = 'for')                         AS goals_for,
       COUNT_IF(side = 'against')                     AS goals_against,
       COUNT_IF(side = 'for') - COUNT_IF(side = 'against') AS diff
FROM team_events
WHERE type_desc_key = 'goal'
GROUP BY 1
ORDER BY game_time = 'OT', game_time;